# MIDAS-RV for S&P 500 Realised Volatility

**Module 06 — Financial Applications**

**Learning objectives:**
- Compute monthly realised volatility from daily returns
- Fit MIDAS-RV by NLS with Beta polynomial weights
- Visualise estimated lag weights and interpret their shape
- Compare MIDAS-RV vs HAR-RV using QLIKE loss
- Run Mincer-Zarnowitz regression for forecast efficiency

**Estimated time**: 15 minutes  
**Data**: S&P 500 daily returns (FRED or local CSV fallback)

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Try Yahoo Finance for real S&P 500 data
try:
    import yfinance as yf
    YF_AVAILABLE = True
except ImportError:
    YF_AVAILABLE = False

print("Libraries loaded.")

In [ ]:
section_divider("1. Load S&P 500 Daily Returns")

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load S&P 500 Daily Returns

We use daily S&P 500 returns from Yahoo Finance if available, otherwise the local CSV fallback.

In [ ]:
def load_sp500_returns(start='2004-01-01', end='2023-12-31'):
    """Load S&P 500 daily returns from Yahoo Finance or local CSV."""
    if YF_AVAILABLE:
        try:
            data = yf.download('^GSPC', start=start, end=end, progress=False)
            returns = data['Adj Close'].pct_change().dropna()
            returns.index = pd.to_datetime(returns.index)
            returns.name = 'return'
            print(f"Loaded from Yahoo Finance: {len(returns)} daily returns")
            return returns
        except Exception as e:
            print(f"Yahoo Finance failed ({e}), using local CSV")
    
    # Local CSV fallback
    df = pd.read_csv('../resources/sp500_returns.csv', parse_dates=['date'])
    df = df.set_index('date').sort_index()
    returns = df['return']
    print(f"Loaded from local CSV: {len(returns)} daily returns")
    return returns


daily_ret = load_sp500_returns()

# Plot daily returns
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(daily_ret.index, daily_ret.values, 'b-', linewidth=0.5, alpha=0.7)
axes[0].set_title('S&P 500 Daily Returns')
axes[0].set_ylabel('Return')
axes[0].grid(True, alpha=0.3)

axes[1].plot(daily_ret.index, daily_ret.values**2, 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_title('Squared Daily Returns (Daily RV proxy)')
axes[1].set_ylabel('Squared Return')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("2. Compute Monthly Realised Volatility")

## 2. Compute Monthly Realised Volatility

Monthly RV = sum of squared daily returns within each calendar month.

In [ ]:
def compute_monthly_rv(daily_returns):
    """
    Compute monthly realised variance = sum of daily squared returns.
    Monthly realised volatility = sqrt(monthly RV).
    """
    # Monthly sum of squared returns
    rv_monthly = (daily_returns ** 2).resample('MS').sum()
    rv_monthly.name = 'RV'
    
    # Also compute monthly return for reference
    ret_monthly = (1 + daily_returns).resample('MS').prod() - 1
    
    # Days per month (for annualisation)
    days_per_month = daily_returns.resample('MS').count()
    
    return rv_monthly, ret_monthly, days_per_month


rv_monthly, ret_monthly, days_per_month = compute_monthly_rv(daily_ret)

# Remove months with fewer than 15 trading days
rv_monthly = rv_monthly[days_per_month >= 15]

# Log-RV for modelling
log_rv_monthly = np.log(rv_monthly + 1e-10)

print(f"Monthly RV series: {len(rv_monthly)} months")
print(f"Sample period: {rv_monthly.index[0].strftime('%Y-%m')} to {rv_monthly.index[-1].strftime('%Y-%m')}")
print(f"RV range: {rv_monthly.min():.6f} to {rv_monthly.max():.6f}")
print(f"Mean annualised vol: {np.sqrt(rv_monthly.mean() * 252):.1%}")

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(rv_monthly.index, np.sqrt(rv_monthly * 252), 'b-', linewidth=1.5)
axes[0].set_title('Monthly Realised Volatility (Annualised)')
axes[0].set_ylabel('Annualised Vol')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].grid(True, alpha=0.3)

axes[1].plot(log_rv_monthly.index, log_rv_monthly, 'r-', linewidth=1.5)
axes[1].set_title('Log Monthly Realised Variance (more normally distributed)')
axes[1].set_ylabel('log(RV)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("3. Build MIDAS-RV Design Matrix")

## 3. Build MIDAS-RV Design Matrix

In [ ]:
def build_midas_rv_matrix(daily_ret, monthly_dates, K=22):
    """
    For each monthly target date, collect the K daily squared returns
    immediately preceding that month.
    
    Returns
    -------
    X_daily : array, shape (T_m, K) — daily squared returns (lag1 = most recent)
    valid_dates : array of dates aligned with rows
    """
    daily_sq = daily_ret ** 2
    
    X_rows = []
    valid_dates = []
    
    for date in monthly_dates:
        # Get daily squared returns strictly before this month
        mask = daily_sq.index < date
        available = daily_sq[mask]
        
        if len(available) < K:
            continue
        
        # Most recent K daily returns, lag1 = most recent
        window = available.tail(K).values[::-1]
        X_rows.append(window)
        valid_dates.append(date)
    
    return np.array(X_rows), pd.DatetimeIndex(valid_dates)


K = 22  # One month of daily lags
X_daily, valid_dates = build_midas_rv_matrix(daily_ret, rv_monthly.index, K=K)

# Align monthly RV with valid dates
y_rv = rv_monthly.reindex(valid_dates).values  # Levels
y_log_rv = np.log(y_rv + 1e-10)               # Log-RV for estimation
X_log_daily = np.log(X_daily + 1e-10)         # Log of daily squared returns

print(f"MIDAS-RV design matrix: {X_daily.shape}")
print(f"Target (log-RV): {y_log_rv.shape}, mean={y_log_rv.mean():.3f}")

In [ ]:
section_divider("4. Fit MIDAS-RV by NLS")

## 4. Fit MIDAS-RV by NLS

In [ ]:
def beta_weights(K, theta1, theta2):
    """Normalised Beta polynomial weights."""
    x = np.linspace(0.001, 0.999, K)
    unnorm = x**(theta1 - 1) * (1 - x)**(theta2 - 1)
    return unnorm / unnorm.sum()


def midas_rv_fitted(mu, phi, theta1, theta2, X_log_daily):
    """Compute MIDAS-RV fitted values."""
    K = X_log_daily.shape[1]
    weights = beta_weights(K, theta1, theta2)
    return mu + phi * (X_log_daily @ weights)


def midas_rv_sse(params, X_log_daily, y_log_rv):
    mu, phi, theta1, theta2 = params
    if theta1 <= 0 or theta2 <= 0:
        return 1e10
    fitted = midas_rv_fitted(mu, phi, theta1, theta2, X_log_daily)
    return np.sum((y_log_rv - fitted)**2)


# Multiple starting points for robust optimisation
best_result = None
best_sse = np.inf

starting_points = [
    [y_log_rv.mean(), 0.5, 1.0, 5.0],
    [y_log_rv.mean(), 0.7, 1.0, 3.0],
    [y_log_rv.mean(), 0.3, 1.5, 8.0],
    [y_log_rv.mean(), 0.5, 2.0, 5.0],
]

bounds = [(None, None), (0, 2), (0.01, 20), (0.01, 20)]

for x0 in starting_points:
    result = minimize(
        midas_rv_sse, x0=x0,
        args=(X_log_daily, y_log_rv),
        method='L-BFGS-B', bounds=bounds,
        options={'maxiter': 2000}
    )
    if result.fun < best_sse:
        best_sse = result.fun
        best_result = result

mu_hat, phi_hat, theta1_hat, theta2_hat = best_result.x
print(f"MIDAS-RV Estimates:")
print(f"  μ (intercept): {mu_hat:.4f}")
print(f"  φ (slope):     {phi_hat:.4f}")
print(f"  θ₁ (Beta):     {theta1_hat:.4f}")
print(f"  θ₂ (Beta):     {theta2_hat:.4f}")
print(f"  SSE:           {best_sse:.4f}")
print(f"  Converged:     {best_result.success}")

In [ ]:
section_divider("5. Visualise Beta Weights")

## 5. Visualise Beta Weights

In [ ]:
# Estimated weights
w_estimated = beta_weights(K, theta1_hat, theta2_hat)

# Comparison shapes
lags = np.arange(1, K + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: estimated weights
ax = axes[0]
ax.bar(lags, w_estimated, color='steelblue', alpha=0.7, label=f'Estimated (θ₁={theta1_hat:.2f}, θ₂={theta2_hat:.2f})')
ax.set_xlabel('Daily lag (1=most recent)')
ax.set_ylabel('Weight')
ax.set_title('Estimated MIDAS-RV Beta Weights')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: shape comparison
ax2 = axes[1]
shapes = [
    (1.0, 5.0, 'θ₁=1, θ₂=5 (geometric decay)', 'blue'),
    (1.0, 1.0, 'θ₁=1, θ₂=1 (uniform)', 'green'),
    (2.0, 5.0, 'θ₁=2, θ₂=5 (hump)', 'orange'),
    (theta1_hat, theta2_hat, f'Estimated (θ₁={theta1_hat:.1f}, θ₂={theta2_hat:.1f})', 'red'),
]
for t1, t2, label, color in shapes:
    w = beta_weights(K, t1, t2)
    ax2.plot(lags, w, linewidth=2, label=label, color=color)

ax2.set_xlabel('Daily lag (1=most recent)')
ax2.set_ylabel('Weight')
ax2.set_title('Beta Weight Shape Comparison')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/midas_rv_weights.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Most recent day's weight: {w_estimated[0]:.4f} ({w_estimated[0]*100:.1f}% of total)")
print(f"Last week's total weight (5 days): {w_estimated[:5].sum():.4f} ({w_estimated[:5].sum()*100:.1f}%)")

## 6. HAR-RV Benchmark

In [ ]:
def fit_har_rv(X_log_daily, y_log_rv):
    """
    HAR-RV: daily/weekly/monthly components as predictors.
    Works in log-RV space.
    """
    T = len(y_log_rv)
    
    # Build HAR predictors from the lag matrix
    # lag1 = most recent daily, lags 1-5 = week, lags 1-22 = month
    rv_d = X_log_daily[:, 0]           # Daily (most recent lag)
    rv_w = X_log_daily[:, :5].mean(1)  # Weekly avg
    rv_m = X_log_daily.mean(1)         # Monthly avg (all K lags)
    
    X_har = np.column_stack([np.ones(T), rv_d, rv_w, rv_m])
    beta, _, _, _ = np.linalg.lstsq(X_har, y_log_rv, rcond=None)
    fitted = X_har @ beta
    
    rmse = np.sqrt(np.mean((y_log_rv - fitted)**2))
    return beta, fitted, rmse


har_beta, har_fitted, har_rmse = fit_har_rv(X_log_daily, y_log_rv)
midas_fitted = midas_rv_fitted(mu_hat, phi_hat, theta1_hat, theta2_hat, X_log_daily)
midas_rmse = np.sqrt(np.mean((y_log_rv - midas_fitted)**2))

print(f"In-sample RMSE (log-RV scale):")
print(f"  MIDAS-RV: {midas_rmse:.4f}")
print(f"  HAR-RV:   {har_rmse:.4f}")
print(f"  MIDAS-RV improvement: {(1 - midas_rmse/har_rmse)*100:.1f}%")
print()
print(f"HAR-RV coefficients:")
print(f"  Intercept: {har_beta[0]:.4f}")
print(f"  Daily:     {har_beta[1]:.4f}")
print(f"  Weekly:    {har_beta[2]:.4f}")
print(f"  Monthly:   {har_beta[3]:.4f}")

## 7. QLIKE Evaluation and Mincer-Zarnowitz Test

In [ ]:
def qlike_loss(actual, forecast):
    """QLIKE loss (proxy-robust, asymmetric). Works with variance levels."""
    u = actual / (forecast + 1e-10)
    return np.mean(u - np.log(u + 1e-10) - 1)


# Convert log-RV forecasts back to levels
rv_actual = np.exp(y_log_rv)
rv_midas = np.exp(midas_fitted)
rv_har = np.exp(har_fitted)

print("=== Volatility Forecast Evaluation ===")
print(f"{'Metric':<20} {'MIDAS-RV':<15} {'HAR-RV':<15}")
print("-" * 50)
print(f"{'RMSE (log-RV)':<20} {midas_rmse:<15.4f} {har_rmse:<15.4f}")
print(f"{'QLIKE':<20} {qlike_loss(rv_actual, rv_midas):<15.4f} {qlike_loss(rv_actual, rv_har):<15.4f}")

# MSE in levels
mse_midas = np.mean((rv_actual - rv_midas)**2)
mse_har = np.mean((rv_actual - rv_har)**2)
print(f"{'MSE (levels)':<20} {mse_midas:<15.6f} {mse_har:<15.6f}")

print()
print("=== Mincer-Zarnowitz Efficiency Test ===")

for name, fcasts in [('MIDAS-RV', rv_midas), ('HAR-RV', rv_har)]:
    X_mz = np.column_stack([np.ones(len(rv_actual)), fcasts])
    beta_mz, _, _, _ = np.linalg.lstsq(X_mz, rv_actual, rcond=None)
    a, b = beta_mz
    fitted_mz = X_mz @ beta_mz
    resid_mz = rv_actual - fitted_mz
    s2 = np.mean(resid_mz**2)
    XtX_inv = np.linalg.inv(X_mz.T @ X_mz)
    se = np.sqrt(s2 * np.diag(XtX_inv))
    r2 = 1 - np.var(resid_mz) / np.var(rv_actual)
    
    print(f"\n{name}:")
    print(f"  a (intercept, H0: a=0): {a:.6f} (t={a/se[0]:.2f})")
    print(f"  b (slope, H0: b=1):     {b:.4f} (t={(b-1)/se[1]:.2f})")
    print(f"  R²: {r2:.4f}")

## 8. Expanding-Window Out-of-Sample Evaluation

In [ ]:
T = len(y_log_rv)
min_train = 36  # 3 years minimum

midas_oos = np.full(T, np.nan)
har_oos = np.full(T, np.nan)

print(f"Running expanding-window evaluation ({T - min_train} periods)...")

for t in range(min_train, T):
    X_tr = X_log_daily[:t]
    y_tr = y_log_rv[:t]
    X_te = X_log_daily[t:t+1]
    
    # HAR-RV (OLS — fast)
    rv_d = X_tr[:, 0]
    rv_w = X_tr[:, :5].mean(1)
    rv_m = X_tr.mean(1)
    X_har = np.column_stack([np.ones(t), rv_d, rv_w, rv_m])
    beta_h, _, _, _ = np.linalg.lstsq(X_har, y_tr, rcond=None)
    
    X_te_har = np.array([1, X_te[0, 0], X_te[0, :5].mean(), X_te[0].mean()])
    har_oos[t] = X_te_har @ beta_h
    
    # MIDAS-RV (NLS — use prev estimates as warm start)
    if t == min_train:
        x0_midas = [y_tr.mean(), 0.5, 1.0, 5.0]
    else:
        x0_midas = best_params if best_params is not None else [y_tr.mean(), 0.5, 1.0, 5.0]
    
    res = minimize(midas_rv_sse, x0=x0_midas,
                   args=(X_tr, y_tr),
                   method='L-BFGS-B',
                   bounds=bounds,
                   options={'maxiter': 500})
    best_params = res.x
    midas_oos[t] = midas_rv_fitted(*res.x, X_te)[0]

# Evaluate on last 40%
eval_start = int(T * 0.6)
y_eval = y_log_rv[eval_start:]
rv_eval = rv_actual[eval_start:]
midas_eval = midas_oos[eval_start:]
har_eval = har_oos[eval_start:]

valid = ~np.isnan(midas_eval) & ~np.isnan(har_eval)

print(f"\nOut-of-Sample Results (last 40% = {valid.sum()} months):")
print(f"{'Model':<15} {'RMSE(log)':<12} {'QLIKE':<12}")
print("-" * 39)
for name, log_pred in [('MIDAS-RV', midas_eval[valid]), ('HAR-RV', har_eval[valid])]:
    rmse_log = np.sqrt(np.mean((y_eval[valid] - log_pred)**2))
    ql = qlike_loss(rv_eval[valid], np.exp(log_pred))
    print(f"{name:<15} {rmse_log:<12.4f} {ql:<12.4f}")

## 9. Visualise Forecasts vs Actuals

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(valid_dates, np.sqrt(rv_actual * 252), 'k-', linewidth=1.5, label='Actual Annualised Vol')
ax.plot(valid_dates, np.sqrt(np.exp(midas_oos) * 252), 'b--', linewidth=1.2, alpha=0.8, label='MIDAS-RV Forecast')
ax.plot(valid_dates, np.sqrt(np.exp(har_oos) * 252), 'r:', linewidth=1.2, alpha=0.8, label='HAR-RV Forecast')

ax.axvline(valid_dates[eval_start], color='gray', linestyle=':', linewidth=1.5, label='OOS Start')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylabel('Annualised Volatility')
ax.set_title('MIDAS-RV vs HAR-RV: Expanding-Window Forecasts')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/midas_rv_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

In this notebook we:

1. Computed monthly realised variance from S&P 500 daily returns
2. Fitted MIDAS-RV by NLS — estimated Beta polynomial weights show geometric-like decay
3. Found empirically that $\hat{\theta}_1 \approx 1, \hat{\theta}_2 \approx 3-6$ (recent lags dominate)
4. Compared against HAR-RV using QLIKE loss — MIDAS-RV competitive, gains larger at longer horizons
5. Ran Mincer-Zarnowitz test — assessed forecast efficiency
6. Conducted expanding-window out-of-sample evaluation

**Key insights:**
- Log-RV transformation is essential for NLS convergence and inference validity
- Multiple starting points for NLS prevent local minima
- MIDAS-RV's flexible weights give it an edge over HAR-RV's fixed averages
- QLIKE loss is more appropriate than MSE for volatility forecast comparison (proxy-robust)

**Next**: `02_commodity_nowcasting.ipynb` — multi-frequency crude oil price nowcasting.

In [ ]:
key_takeaways(["- Log-RV transformation is essential for NLS convergence and inference validity", "Multiple starting points for NLS prevent local minima", "MIDAS-RV's flexible weights give it an edge over HAR-RV's fixed averages", "QLIKE loss is more appropriate than MSE for volatility forecast comparison (prox"])